# 수면 자세 로컬 분류 모델 훈련

**전략**: MobileNetV2(ImageNet 사전학습) 특징 추출 + SVM 분류기

- 136장 소규모 데이터셋에 최적 (fine-tuning 불필요)
- Gemini 대비 inference 100배+ 빠름, API 비용 없음
- 신뢰도 < 60% 인 경우에만 Gemini fallback
- 훈련 완료 후 `backend/posture_model.pkl` 저장

In [ ]:
%pip install tensorflow scikit-learn joblib pillow numpy matplotlib seaborn tqdm --quiet

## 1. 설정

In [ ]:
import io, zipfile
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm

from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

DATASET_ZIP    = "dataset_copy.zip"
MODEL_OUT      = "../backend/posture_model.pkl"
IMG_SIZE       = (224, 224)
CONF_THRESHOLD = 0.60   # 이 신뢰도 미만이면 Gemini fallback

FOLDER_TO_LABEL = {
    "left":   "Lateral_L",
    "right":  "Lateral_R",
    "prone":  "Prone",
    "supine": "Supine",
}

print(f"TensorFlow {tf.__version__}")
print(f"GPU: {len(tf.config.list_physical_devices('GPU'))}개")

## 2. 데이터 로드

In [ ]:
def load_dataset(zip_path):
    images, labels, filenames = [], [], []
    with zipfile.ZipFile(zip_path) as zf:
        names = [n for n in zf.namelist() if n.lower().endswith(".png")]
        for name in tqdm(names, desc="이미지 로드"):
            parts = Path(name).parts
            if len(parts) < 3:
                continue
            folder = parts[1].lower()
            if folder not in FOLDER_TO_LABEL:
                continue
            img_bytes = zf.read(name)
            img = Image.open(io.BytesIO(img_bytes)).convert("RGB").resize(IMG_SIZE)
            images.append(np.array(img))
            labels.append(FOLDER_TO_LABEL[folder])
            filenames.append(Path(name).name)
    return np.array(images), np.array(labels), filenames

X_raw, y, fnames = load_dataset(DATASET_ZIP)

print(f"\n총 {len(X_raw)}장 로드")
for lbl in sorted(np.unique(y)):
    print(f"  {lbl:12s}: {(y == lbl).sum()}장")

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, lbl in zip(axes, ["Lateral_L", "Lateral_R", "Prone", "Supine"]):
    idx = np.where(y == lbl)[0][0]
    ax.imshow(X_raw[idx])
    ax.set_title(lbl, fontsize=10)
    ax.axis("off")
plt.suptitle("클래스별 샘플")
plt.tight_layout()
plt.show()

## 3. MobileNetV2 특징 추출

ImageNet 사전학습 가중치로 frozen. Global Average Pooling → 이미지당 **1280차원** 벡터.

In [ ]:
extractor = MobileNetV2(
    weights="imagenet",
    include_top=False,
    pooling="avg",
    input_shape=(*IMG_SIZE, 3),
)
extractor.trainable = False

print(f"특징 추출기 파라미터: {extractor.count_params():,}개 (frozen)")

X_prep = preprocess_input(X_raw.astype(np.float32))
print("특징 추출 중...")
X_feat = extractor.predict(X_prep, batch_size=16, verbose=1)
print(f"완료: {X_feat.shape}  (이미지 수, 특징 차원)")

## 4. SVM 훈련 + 5-Fold 교차 검증

In [ ]:
le = LabelEncoder()
y_enc = le.fit_transform(y)
print(f"클래스 순서: {list(le.classes_)}")

clf = SVC(kernel="rbf", C=10, gamma="scale", probability=True, random_state=42)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(clf, X_feat, y_enc, cv=skf, scoring="accuracy")

print("\n[5-Fold CV 결과]")
for i, s in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {s*100:.1f}%")
print(f"  평균: {cv_scores.mean()*100:.1f}% ± {cv_scores.std()*100:.1f}%")

clf.fit(X_feat, y_enc)
print("\n최종 모델 학습 완료")

## 5. 성능 평가

In [ ]:
y_pred = clf.predict(X_feat)
y_prob = clf.predict_proba(X_feat)
conf   = y_prob.max(axis=1)

print("=" * 50)
print(f"  Training 정확도: {accuracy_score(y_enc, y_pred)*100:.1f}%")
print("=" * 50)
print()
print(classification_report(
    le.inverse_transform(y_enc),
    le.inverse_transform(y_pred)
))

low_conf = (conf < CONF_THRESHOLD).sum()
print(f"[신뢰도] 평균={conf.mean()*100:.1f}%  최소={conf.min()*100:.1f}%")
print(f"  {CONF_THRESHOLD*100:.0f}% 미만 (Gemini fallback 대상): {low_conf}장 ({low_conf/len(conf)*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(y_enc, y_pred, labels=range(len(le.classes_)))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[0])
axes[0].set_title("Count")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[1], vmin=0, vmax=1)
axes[1].set_title("Normalized")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")

plt.suptitle("Confusion Matrix — 로컬 MobileNetV2+SVM", fontsize=13)
plt.tight_layout()
plt.savefig("confusion_matrix_local.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. 모델 저장

In [ ]:
model_data = {
    "svm":            clf,
    "label_encoder":  le,
    "img_size":       IMG_SIZE,
    "conf_threshold": CONF_THRESHOLD,
    "feature_model":  "MobileNetV2-imagenet-gap",
}

out_path = Path(MODEL_OUT)
out_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model_data, out_path)

print(f"저장 완료: {out_path}")
print(f"파일 크기: {out_path.stat().st_size // 1024} KB")

## 7. 추론 테스트

In [ ]:
import random

def classify_local(img_source, model_data, feature_extractor):
    if isinstance(img_source, (str, Path)):
        img = Image.open(img_source).convert("RGB")
    else:
        img = Image.open(io.BytesIO(img_source)).convert("RGB")
    img  = img.resize(model_data["img_size"])
    x    = preprocess_input(np.array(img, dtype=np.float32)[None])
    feat = feature_extractor.predict(x, verbose=0)
    prob     = model_data["svm"].predict_proba(feat)[0]
    pred_idx = int(np.argmax(prob))
    label    = model_data["label_encoder"].inverse_transform([pred_idx])[0]
    return label, float(prob[pred_idx])

test_indices = random.sample(range(len(fnames)), min(8, len(fnames)))
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

with zipfile.ZipFile(DATASET_ZIP) as zf:
    all_names = zf.namelist()
    for ax, idx in zip(axes, test_indices):
        folder = [k for k, v in FOLDER_TO_LABEL.items() if v == y[idx]][0]
        zip_path = next(n for n in all_names if folder in n and fnames[idx] in n)
        img_bytes = zf.read(zip_path)
        posture, conf_val = classify_local(img_bytes, model_data, extractor)
        correct = posture == y[idx]
        img = Image.open(io.BytesIO(img_bytes)).resize((180, 270))
        ax.imshow(img)
        ax.set_title(
            f"True: {y[idx]}\nPred: {posture} ({conf_val*100:.0f}%)",
            fontsize=8, color="green" if correct else "red"
        )
        ax.axis("off")

for ax in axes[len(test_indices):]:
    ax.axis("off")

plt.suptitle("로컬 모델 추론 샘플", fontsize=12)
plt.tight_layout()
plt.show()